# Lab de la clase 6: de bronze a gold, silver y gold con tus funciones

En la clase 5 ingestaste las tres fuentes a tu propio bronze. Ahora escribes la lógica que lleva esos datos hasta la tabla que responde la pregunta del curso, ¿la lluvia cambia los viajes en NYC? El pipeline sigue el patrón *medallion*: bronze guarda cada fuente como llegó, silver la limpia y la enriquece, y gold la resume.

La infraestructura ya está lista: la sesión de Spark, la muestra de datos, las rutas y las pruebas. Tú escribes solo la lógica, dentro de huecos marcados con `# TU CÓDIGO AQUÍ`. Debajo de cada ejercicio hay una prueba que te dice al instante si tu código está bien, con tres posibles respuestas: `OK` si pasó todo, `PENDIENTE` si todavía queda un hueco sin resolver y `REVISAR`, con una pista, si tu código corre pero algo no coincide con lo esperado.

Cada ejercicio trae además pistas plegables, ordenadas de la más general a la más concreta. Ábrelas solo cuando las necesites y de una en una. Cada celda de código empieza con un comentario `# Celda N`: es su número dentro de este notebook, y el texto lo usa cuando te manda a una celda concreta.

Los ejercicios son cuatro. En el 1 limpias y tipas los viajes. En el 2 quitas los duplicados usando la huella, que es un *shuffle*. En el 3 unes las zonas con *broadcast* y en el 4 construyes gold con promedios ponderados. Al final comparas tu pipeline con el del repositorio y lo corres desde la terminal. La huella de cada fila (`_hash_fila`) y el clima que usa la cadena final salen de lo que hiciste en la clase 5: la huella es la misma que escribiste en el ejercicio 1 de ese lab, y el clima es el que ingestaste en tu bronze. Si no hiciste esa ingesta, la comparación final usa el clima del repositorio y llegas a ella de todos modos.

Lo que escribes aquí es lo mismo que después corre en Airflow en local y luego en Glue, sin reescribirlo: por eso cada función recibe y devuelve DataFrames, con los mismos nombres que las del pipeline de `src/`.

> **Antes de empezar.** En esta máquina solo cabe un proceso de Spark a la vez. Si ya abriste otro notebook de la clase y dejaste Spark encendido, ciérralo primero (cada notebook termina con una celda que apaga Spark) o reinicia el kernel del otro notebook.

## 0. Preparación: el entorno correcto, tu bucket y Spark encendido

Es la misma preparación de los notebooks anteriores. Las celdas ya vienen escritas y las ejecutas sin cambiarlas.

### 0.1 Verificar el kernel y Java

El kernel es el programa Python que ejecuta las celdas. Si eliges uno equivocado, las librerías no estarán instaladas. Elige siempre **Python (bigdata)**, que es el entorno `.venv` del proyecto. No elijas los kernels que se llaman *Glue PySpark*, *Glue Spark* o *SparkMagic*: abren una sesión de AWS Glue en la nube y esa sesión cobra mientras está abierta. La Celda 1 imprime qué Python usas y si Java está disponible, porque Spark corre sobre la máquina virtual de Java.

In [ ]:
# Celda 1
# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup
# en ~/.local/opt/java17. Si no está en el PATH, lo buscamos ahí y definimos JAVA_HOME antes de usar Spark.
import os, shutil
from pathlib import Path
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

import json, re, subprocess, sys, time, contextlib, gc
from io import StringIO

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto (la ruta no contiene .venv). "
          "Elige el kernel «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("Java:", (java.stderr or java.stdout).splitlines()[0])
except FileNotFoundError:
    print("AVISO: no se encontró Java en el PATH; Spark no podrá arrancar.")

import pandas, pyarrow, pyspark
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyspark", pyspark.__version__)

### 0.2 Tu bucket, armado desde tu cuenta de AWS

Cada estudiante tiene su propio lake en S3. Su nombre no se escribe a mano: se arma con un patrón fijo, `<prefijo>-lake-<número de cuenta>-<región>`, igual que hace la función `lake_bucket` de `scripts/lib.sh` cuando corres un `make`. El número de cuenta se lo preguntamos a AWS con las credenciales del Code Editor (es lo mismo que `aws sts get-caller-identity`). La Celda 2 lo arma, lo imprime y comprueba que existe y que puedes leerlo.

In [ ]:
# Celda 2
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-2"

# Raíz del repositorio: la carpeta que contiene pyproject.toml, subiendo desde donde está este notebook.
RAIZ = Path.home() / "bigdata"
for carpeta in [Path.cwd(), *Path.cwd().parents]:
    if (carpeta / "pyproject.toml").exists():
        RAIZ = carpeta
        break
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline import ...»

# El prefijo viene del archivo .env del repo (lo escribe «make setup»); si no existe, es «bigdata».
prefijo = os.environ.get("PREFIX")
if not prefijo and (RAIZ / ".env").exists():
    m = re.search(r"^PREFIX=(.+)$", (RAIZ / ".env").read_text(), re.M)
    if m:
        prefijo = m.group(1).strip()
if not prefijo:
    prefijo = "bigdata"

cuenta = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"{prefijo}-lake-{cuenta}-{REGION}"
print("Cuenta AWS :", cuenta)
print("Tu bucket  :", BUCKET)

s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print("Acceso al bucket: correcto")
except ClientError as e:
    print("AVISO: no se pudo leer el bucket:", e.response["Error"]["Code"],
          "- se usará el bucket público del curso para el archivo de enero.")

### 0.3 Los datos de enero, una sola vez, en el disco local

Spark lee archivos del disco de esta máquina, así que primero hay que traer el archivo de viajes de enero desde tu bucket (`bronze/_crudo/fhvhv/mes=2024-01/`, y si tu lake todavía tiene el diseño anterior, `bronze/fhvhv/mes=2024-01/`). Se guarda fuera del repositorio, en la carpeta `~/datos_clase05`, para que nunca llegue a un commit por accidente. Si la carpeta ya tiene el archivo, la celda no vuelve a descargarlo. Si tu bucket no lo tiene, usa la copia del bucket público del curso. Con las zonas pasa lo mismo, y si tampoco están en tu bucket se bajan del sitio de la TLC, que es de donde las toma `bronze.py`. La Celda 3 hace esa descarga.

In [ ]:
# Celda 3
from botocore import UNSIGNED
from botocore.config import Config
import urllib.request

DATOS = Path(os.environ.get("CLASE05_DATOS", Path.home() / "datos_clase05"))
RUTA_DIR_ENERO = DATOS / "bronze" / "fhvhv" / "mes=2024-01"
RUTA_ENERO = RUTA_DIR_ENERO / "fhvhv_tripdata_2024-01.parquet"
RUTA_ZONAS = DATOS / "bronze" / "taxi_zones" / "taxi_zone_lookup.csv"
MB = 1024 * 1024

from pipeline import bronze as bronze_pipeline   # trae URL_ZONAS, url_clima y las funciones de ingesta
from pipeline import io as pio                   # trae las rutas del lake: crudo, lotes, etc.

# El archivo crudo vive en bronze/_crudo/... (diseño nuevo). Si tu lake todavía tiene el diseño anterior
# (bronze/<fuente>/mes=.../<archivo>), la descarga prueba también esa ruta.
NOMBRE_ENERO = pio.nombre_viajes("2024-01")
CLAVES_ENERO = [pio.ruta_crudo("bronze", pio.VIAJES, NOMBRE_ENERO, "2024-01"),
                pio.ruta_anterior("bronze", pio.VIAJES, NOMBRE_ENERO, "2024-01")]
CLAVES_ZONAS = [pio.ruta_crudo("bronze", pio.ZONAS, pio.NOMBRE_ZONAS),
                pio.ruta_anterior("bronze", pio.ZONAS, pio.NOMBRE_ZONAS)]


def traer(claves, destino, url_respaldo=None):
    """Descarga a destino el primer objeto que exista entre `claves`, si destino todavía no existe.

    Busca primero en tu bucket, luego en el bucket público del curso y, si se indica, en una URL de respaldo.
    """
    if destino.exists():
        print(f"Ya existe, no se descarga: {destino}  ({destino.stat().st_size / MB:,.1f} MiB)")
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_name(destino.name + ".parcial")   # evita dejar un archivo a medias
    t0 = time.perf_counter()
    origen = None
    for clave in claves:
        try:
            s3.download_file(BUCKET, clave, str(parcial))
            origen = f"s3://{BUCKET}/{clave}"
            break
        except ClientError:
            pass
    if origen is None and url_respaldo:
        urllib.request.urlretrieve(url_respaldo, parcial)
        origen = url_respaldo
    if origen is None:
        publico = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))
        for clave in claves:
            try:
                publico.download_file("bigdata-udem-2026-publico", "paquete/" + clave, str(parcial))
                origen = f"s3://bigdata-udem-2026-publico/paquete/{clave}"
                break
            except ClientError:
                pass
    if origen is None:
        raise FileNotFoundError(f"No se encontró ninguno de estos objetos en tu bucket ni en el público: {claves}")
    parcial.rename(destino)
    print(f"Descargado desde {origen}\n  -> {destino}  ({destino.stat().st_size / MB:,.1f} MiB en {time.perf_counter() - t0:.1f} s)")


traer(CLAVES_ENERO, RUTA_ENERO)
traer(CLAVES_ZONAS, RUTA_ZONAS, url_respaldo=bronze_pipeline.URL_ZONAS)

### 0.4 Dos utilidades pequeñas

Para no repetir código, la Celda 4 define un cronómetro que imprime cuánto tardó un bloque y una función `fmt` que escribe los números grandes con espacios entre los miles, como en el resto del curso.

In [ ]:
# Celda 4
TIEMPOS = {}


@contextlib.contextmanager
def cronometro(etiqueta):
    """Mide un bloque, lo imprime y lo guarda en TIEMPOS."""
    t0 = time.perf_counter()
    yield
    TIEMPOS[etiqueta] = time.perf_counter() - t0
    print(f"[{etiqueta}] {TIEMPOS[etiqueta]:.2f} s")


def fmt(n):
    """19663930 -> '19 663 930'"""
    return f"{n:,}".replace(",", " ")

### 0.5 Arrancar Spark con la misma configuración del pipeline

No vamos a inventar una configuración aparte para el notebook: usamos la misma función `get_spark("local")` de `src/pipeline/engine.py` que usa el pipeline. Crea una sesión `local[*]`, que significa que todo vive en un solo proceso de Java: el driver y un único ejecutor, que tiene tantos hilos como núcleos tenga la máquina (dos en el Code Editor). No hay dos ejecutores: hay uno con dos hilos, y cada hilo corre una tarea a la vez. El driver tiene 2 GB de memoria, la zona horaria de la sesión queda en UTC y los reemplazos de particiones son dinámicos.

Después de arrancar, la Celda 5 imprime `defaultParallelism`, el número de tareas que Spark puede ejecutar al mismo tiempo, y el enlace a la Spark UI, la página donde Spark muestra cada trabajo, etapa y tarea mientras corre.

In [ ]:
# Celda 5
# La Spark UI se ve a través del proxy del Code Editor, que agrega el prefijo /codeeditor/default/ports/4040.
# Spark necesita saberlo para armar sus enlaces, estilos e imágenes con ese prefijo; si no, la página
# carga sin diseño. Se indica antes de arrancar Spark, porque se lee al encender la JVM.
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.proxyBase=/codeeditor/default/ports/4040 pyspark-shell"

from pyspark.sql import functions as F, DataFrame, Window
from pipeline.engine import get_spark

with cronometro("arrancar Spark"):
    spark = get_spark("local")

print("Versión de Spark :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Memoria del driver:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("defaultParallelism (tareas simultáneas):", spark.sparkContext.defaultParallelism)


def url_spark_ui():
    """Enlace a la Spark UI. Dentro del Code Editor pasa por el proxy del Space, igual que Airflow."""
    local = spark.sparkContext.uiWebUrl            # p. ej. http://10.0.0.5:4040
    puerto = local.rsplit(":", 1)[-1].strip("/")
    meta = Path("/opt/ml/metadata/resource-metadata.json")
    if meta.exists():
        try:
            datos = json.loads(meta.read_text())
            r = boto3.client("sagemaker", region_name=REGION).describe_space(
                DomainId=datos["DomainId"], SpaceName=datos["SpaceName"])
            base = re.match(r"https://[^/]+", r["Url"]).group(0)
            return f"{base}/codeeditor/default/ports/{puerto}/jobs/"   # /jobs/ directo: la raíz redirige mal detrás del proxy
        except Exception as e:                      # sin permiso o fuera del Code Editor
            print("(no se pudo consultar el Space:", type(e).__name__, ")")
    return f"{local}   (en el Code Editor: https://<id-del-space>.studio.{REGION}.sagemaker.aws/codeeditor/default/ports/{puerto}/)"


print("Spark UI:", url_spark_ui())

**Qué observar.** `defaultParallelism` es el número de hilos del único ejecutor de esta máquina; esa cifra manda cuántas tareas avanzan a la vez. Abre el enlace de la Spark UI en otra pestaña y déjalo abierto: la pestaña *Jobs* está vacía porque Spark todavía no ha hecho nada, y se irá llenando conforme ejecutes celdas. La UI solo existe mientras la sesión de Spark está viva.

## 1. El medallion paso a paso, y qué guarda el bronze que ingestaste

Un pipeline separa sus datos en tres capas según cuánto se han transformado:

- **Bronze** guarda cada fuente tal como llega. Se organiza en una carpeta por fuente y por mes, y nada de lo que entra se modifica ni se borra. Si algo sale mal más adelante, bronze permite reprocesar sin volver a pedirle los datos a la fuente.
- **Silver** limpia y enriquece: convierte tipos, descarta los viajes imposibles (distancias de cero millas, duraciones de horas o negativas) y une las zonas y el clima. El `silver.py` del repositorio todavía **no** quita duplicados: esa pieza es tu ejercicio 2, y entra al pipeline en esta clase.
- **Gold** agrega: una fila por día, zona y empresa, con los números que responden la pregunta.

Esto es lo que dejó tu ingesta de la clase 5. Cada ejecución creó un *lote*, una carpeta nueva con un identificador único, y nunca reemplazó a los anteriores. Ejemplo con el clima de enero:

```text
bronze/
  _crudo/clima/mes=2024-01/lote=20260117T143005Z_a3f9c1/open_meteo_2024-01.json   el archivo tal como lo entregó la fuente
  clima/mes=2024-01/lote=20260117T143005Z_a3f9c1/                                  Parquet con una fila por día y cinco columnas de metadatos
  clima/mes=2024-01/lote=20260118T091210Z_07be44/                                  una segunda ingesta del mismo mes: otro lote, el anterior no se borra
```

Dentro del lote, cada fila lleva sus columnas originales y estas cinco columnas de metadatos:

| Columna | Qué guarda | Para qué sirve |
|---|---|---|
| `_fecha_ingesta` | El momento exacto, en UTC, en que se ingestó el lote. | Saber qué tan fresco es el dato y ordenar lotes en el tiempo. |
| `_id_lote` | El identificador del lote: la fecha UTC de la ingesta y 6 caracteres hexadecimales al azar, por ejemplo `20260117T143005Z_a3f9c1`. | Distinguir una ingesta de otra. Como empieza con la fecha, ordenarlo como texto lo ordena en el tiempo. |
| `_hash_fila` | La huella SHA-256 de la fila: un texto de 64 caracteres que se calcula con todas las columnas originales. | Detectar filas repetidas y filas cambiadas sin comparar columna por columna. Dos filas con los mismos valores dan la misma huella. |
| `_fuente` | La URL de donde se descargó el dato. | Saber de dónde viene cada fila. |
| `_archivo_origen` | El nombre del archivo crudo del que salió la fila. | Volver al crudo y reproducir el resultado. |

La huella se calcula así: cada columna original se convierte a texto, el nulo se reemplaza por el texto `<nulo>` (para que un nulo no se confunda con un texto vacío), los textos se unen con `||` y a ese texto se le aplica SHA-256. Los metadatos no entran en la huella, y por eso el mismo dato ingestado dos veces tiene la misma huella aunque tenga otro lote y otra fecha.

El pipeline de silver lee siempre el lote más reciente que haya terminado de escribirse (el que tiene el marcador `_LOTE_COMPLETO`, un archivo vacío que la ingesta publica al final, cuando el Parquet ya se escribió completo; se usa un marcador propio porque Glue no escribe el `_SUCCESS` de Spark). Por eso silver no duplica los datos aunque reingestes: de todos los lotes que hay en bronze solo lee uno. La huella sirve en los otros dos casos: cuando alguien lee la carpeta del mes entera, con todos los lotes juntos (lo verás con el clima en la Celda 19), y cuando la fuente publica la misma fila dos veces.

## 2. Preparación del lab: una muestra y la forma de las pruebas

Las funciones del lab no corren sobre los 19.6 millones de viajes sino sobre una **muestra determinista del 1 % de enero** (`sample` con semilla 42) repartida en todo el mes. La celda imprime cuántas filas salen exactamente. Así cada prueba tarda segundos y el resultado es el mismo cada vez.

Los datos reales de enero traen muy pocas filas completamente repetidas, y para que el ejercicio de duplicados tenga algo que encontrar la celda repite **a propósito** una fracción de las filas de la muestra (el 2 %). Esa es la única parte inventada del lab. Para esos viajes la columna `_hash_fila` se llena con una huella provisional (`xxhash64`), distinta de la del pipeline, para que el ejercicio 2 se pruebe solo, sin depender del resto.

La función `verificar` ejecuta la prueba de cada ejercicio e imprime una de tres cosas: `OK`, `PENDIENTE` si queda un hueco sin resolver (una línea con `...  # TU CÓDIGO AQUÍ`) o `REVISAR` con una pista. La Celda 6 define `verificar`, y la Celda 8 arma la muestra y la referencia.

In [ ]:
# Celda 6
import inspect
from datetime import date

from pipeline import silver, gold
from pipeline.silver import CARGOS, DISTANCIA_MAX_MI, DURACION_MIN_MIN, DURACION_MAX_MIN, _empresa, clima_diario
from pipeline.gold import COLUMNAS

MESES_LAB = ["2024-01"]
FRACCION_MUESTRA = 0.01
CARPETA_BRONZE = DATOS / "bronze_alumno"            # aquí quedó tu ingesta de la clase 5
TABLA_CLIMA_CLASE5 = "prueba_clima"                 # la tabla de práctica con la que ingestaste el clima en la clase 5

# La huella de cada fila es la que escribiste en la clase 5: la función hash_fila del pipeline hace lo mismo.
calcular_hash_fila = bronze_pipeline.hash_fila


RESULTADO_LAB = {}
HUECO = re.compile(r"(=|^)\s*\.\.\.\s*#\s*TU CÓDIGO AQUÍ", re.M)


def verificar(nombre, funcion, prueba):
    """Corre prueba(funcion) e imprime OK, PENDIENTE o REVISAR con una pista."""
    try:
        fuente = inspect.getsource(funcion)
    except (OSError, TypeError):
        fuente = ""
    if HUECO.search(fuente):
        estado, detalle = "PENDIENTE", "todavía hay un hueco «...  # TU CÓDIGO AQUÍ» sin resolver."
    else:
        try:
            prueba(funcion)
            estado, detalle = "OK", ""
        except AssertionError as e:
            estado, detalle = "REVISAR", str(e)
        except Exception as e:
            estado, detalle = "REVISAR", f"tu función falló con {type(e).__name__}: {str(e)[:300]}"
    RESULTADO_LAB[nombre] = estado
    if detalle:
        print(f"{nombre}: {estado}  {detalle}")
    else:
        print(f"{nombre}: {estado}")

In [ ]:
# Celda 7
def texto_plan(df, modo="simple"):
    """El plan de df como texto (explain() imprime; aquí se captura)."""
    salida = StringIO()
    with contextlib.redirect_stdout(salida):
        df.explain(mode=modo)
    return salida.getvalue()


def mostrar_plan(df, claves=None):
    """Imprime el plan; si se dan claves, solo las líneas que contienen alguna de ellas.
    La línea del FileScan es larguísima: se resume a sus filtros empujados y a las columnas que lee."""
    for linea in texto_plan(df).splitlines():
        if claves is not None:
            contiene = False
            for clave in claves:
                if clave in linea:
                    contiene = True
            if not contiene:
                continue
        if "FileScan" in linea:
            trozos = []
            for patron in (r"PushedFilters: \[.*?\]", r"ReadSchema: .*$"):
                encontrado = re.search(patron, linea)
                if encontrado:
                    trozos.append(encontrado.group(0))
            linea = "FileScan parquet | " + " | ".join(trozos)
        print(linea.strip()[:400])


base_v = spark.read.parquet(str(RUTA_DIR_ENERO))
print("=== Sin filtro: select de 2 columnas ===")
mostrar_plan(base_v.select("PULocationID", "base_passenger_fare"), ["FileScan", "Project"])
print("\n=== Con filtro: where PULocationID = 132 ===")
mostrar_plan(base_v.where(F.col("PULocationID") == 132).select("PULocationID", "base_passenger_fare"),
             ["FileScan", "Filter", "Project"])

In [ ]:
# Celda 8
viajes_lab = spark.read.parquet(str(RUTA_DIR_ENERO))
zonas = spark.read.option("header", True).csv(str(RUTA_ZONAS))

with cronometro("armar la muestra"):
    muestra = viajes_lab.sample(False, FRACCION_MUESTRA, seed=42).cache()
    n_muestra = muestra.count()
repetidas = muestra.sample(False, 0.02, seed=7)
muestra_dup = muestra.unionByName(repetidas)       # la muestra con filas repetidas a propósito

# Huella provisional solo para el ejercicio 2 (la del pipeline se usa al final, en la cadena completa).
columnas_viaje = muestra_dup.columns
muestra_dup_h = muestra_dup.withColumn("_hash_fila", F.xxhash64(*columnas_viaje).cast("string")).cache()
n_dup = muestra_dup_h.count()
print(f"Muestra: {fmt(n_muestra)} viajes (1 % de enero, semilla 42)")
print(f"Muestra con filas repetidas añadidas a propósito: {fmt(n_dup)} filas ({fmt(n_dup - n_muestra)} repetidas)")

# Referencia del pipeline del repo sobre la muestra, para que las pruebas de silver y gold
# no dependan de que hayas resuelto otros ejercicios (el clima viene del archivo de pruebas del repo).
json_pruebas = spark.read.option("multiLine", True).json(str(RAIZ / "tests" / "data" / "open_meteo_sample.json"))
clima_pruebas = clima_diario(bronze_pipeline.clima_a_filas(json_pruebas))
# El silver.py del repositorio no quita duplicados (esa pieza es el ejercicio 2): aquí se quitan a mano con dropDuplicates().
silver_ref = silver.enriquecer(silver.limpiar_viajes(muestra.dropDuplicates(), MESES_LAB), zonas, clima_pruebas).cache()
print("Filas de la referencia de silver:", fmt(silver_ref.count()))


def unir_clima(trips: DataFrame, clima: DataFrame) -> DataFrame:
    """Ya resuelta: une el clima del día de recogida (tabla pequeña, en broadcast)."""
    clima_sel = clima.select(F.col("fecha").alias("pickup_date"), "lluvia_mm", "temp_media_c")
    return trips.join(F.broadcast(clima_sel), "pickup_date", "left")

**Qué observar.** La muestra tiene unos 197 mil viajes y la versión con repetidos tiene unos miles más. La referencia de silver es lo que produce el código del repositorio sobre esos datos y sirve para que las pruebas de silver y gold comparen contra algo fijo. Como el `silver.py` del repositorio no quita duplicados, la celda lo hace a mano con `dropDuplicates()` antes de limpiar; reconstruir esa pieza es el ejercicio 2.

### Ejercicio 1 · Silver: tipar columnas y descartar los viajes imposibles

Escribes tu código en la Celda 9 y la Celda 10 lo verifica.

**Qué hacer.** Completa `limpiar_tipos(trips)`, que recibe los viajes crudos de bronze y devuelve los viajes limpios con las mismas reglas de `silver.limpiar_viajes`. Hay tres huecos. Primero, `duracion`: en bronze `trip_time` viene en **segundos** y silver trabaja en **minutos**. Segundo, `total_pagado`: lo que pagó el pasajero es la suma de los cargos de la lista `CARGOS` (tarifa base, peajes, `bcf`, impuesto, recargo de congestión y cargo de aeropuerto, donde un nulo cuenta como 0) más la propina. Tercero, `condicion`: se conserva el viaje solo si tiene fecha y zona de recogida, una distancia mayor que 0 y menor que `DISTANCIA_MAX_MI`, una tarifa base mayor que 0 y una duración entre `DURACION_MIN_MIN` y `DURACION_MAX_MIN` (ambos extremos incluidos).

**Por qué importa.** Los datos crudos traen viajes de 0 millas, de horas de duración, sin zona. Si no se descartan, los promedios de gold salen distorsionados y nadie se da cuenta. Esta es una transformación estrecha: cada fila se decide sola, sin mirar a las demás, así que escala sin shuffle. La función `_empresa` (que traduce la licencia a Uber, Lyft, etcétera) ya viene resuelta.

<details>
<summary>Pista 1: duración y total pagado</summary>

`trip_time` viene en segundos y la duración se guarda en minutos, como `double`: convierte primero a `double` y divide entre 60. Para `total_pagado` ya tienes calculados los dos ingredientes unas líneas más arriba, `cargos` y `propina`.

</details>

<details>
<summary>Pista 2: la condición</summary>

La condición es una sola expresión que une las reglas con `&`, cada una entre paréntesis: fecha y zona de recogida no nulas (`isNotNull`), distancia mayor que 0 y menor que `DISTANCIA_MAX_MI`, tarifa base mayor que 0, y duración entre `DURACION_MIN_MIN` y `DURACION_MAX_MIN` con los dos extremos incluidos (`between` los incluye). Si la prueba dice que no descartaste un caso, ese caso te dice qué regla falta.

</details>

In [ ]:
# Celda 9
def limpiar_tipos(trips: DataFrame) -> DataFrame:
    """Tipos, duración, empresa, total pagado y filtros de outliers (mismas reglas que silver.limpiar_viajes)."""
    duracion = F.col("trip_time").cast("double") / 60.0
    propina = F.coalesce(F.col("tips").cast("double"), F.lit(0.0))

    cargos = F.lit(0.0)
    for nombre_cargo in CARGOS:
        cargo = F.coalesce(F.col(nombre_cargo).cast("double"), F.lit(0.0))     # un cargo nulo cuenta como 0
        cargos = cargos + cargo
    total_pagado = cargos + propina

    base = trips.select(
        F.to_timestamp("pickup_datetime").alias("pickup_datetime"),
        F.to_timestamp("dropoff_datetime").alias("dropoff_datetime"),
        F.col("PULocationID").cast("int").alias("pu_location_id"),
        F.col("DOLocationID").cast("int").alias("do_location_id"),
        _empresa("hvfhs_license_num").alias("empresa"),
        F.col("trip_miles").cast("double").alias("distancia_mi"),
        duracion.alias("duracion_min"),
        F.col("base_passenger_fare").cast("double").alias("tarifa_base"),
        propina.alias("propina"),
        F.col("driver_pay").cast("double").alias("pago_conductor"),
        total_pagado.alias("total_pagado"),
    )

    condicion = (
        F.col("pickup_datetime").isNotNull()
        & F.col("pu_location_id").isNotNull()
        & (F.col("distancia_mi") > 0)
        & (F.col("distancia_mi") < DISTANCIA_MAX_MI)
        & (F.col("tarifa_base") > 0)
        & F.col("duracion_min").between(DURACION_MIN_MIN, DURACION_MAX_MIN)
    )
    return base.where(condicion).withColumn("pickup_date", F.to_date("pickup_datetime"))

In [ ]:
# Celda 10
COLS_ENTRADA = (
    "hvfhs_license_num string, pickup_datetime string, dropoff_datetime string, "
    "PULocationID int, DOLocationID int, trip_miles double, trip_time int, "
    "base_passenger_fare double, tolls double, bcf double, sales_tax double, "
    "congestion_surcharge double, airport_fee double, tips double, driver_pay double"
)


def _viaje(pickup="2024-01-10 10:00:00", segundos=1200, pu=43, distancia=3.0, tarifa=12.0, licencia="HV0003"):
    return (licencia, pickup, None, pu, 79, distancia, segundos, tarifa, 0.0, 0.3, 1.0, 2.75, None, 2.0, 9.0)


def redondear_total(df):
    return df.withColumn("total_pagado", F.round("total_pagado", 6))


def prueba_e1(f):
    # 1. un viaje normal sobrevive y trae los valores esperados
    filas = f(spark.createDataFrame([_viaje()], COLS_ENTRADA)).collect()
    assert len(filas) == 1, "Un viaje normal (3 mi, 20 min, tarifa 12) debería sobrevivir y no quedó."
    r = filas[0]
    assert abs(r["duracion_min"] - 20.0) < 1e-9, f"duracion_min debería ser 20.0 (1200 segundos) y es {r['duracion_min']}: trip_time viene en segundos."
    assert r["empresa"] == "Uber", "La empresa de HV0003 debe ser Uber (usa _empresa)."
    esperado = 12.0 + 0.0 + 0.3 + 1.0 + 2.75 + 0.0 + 2.0
    assert abs(r["total_pagado"] - esperado) < 1e-9, f"total_pagado debería ser {esperado} (tarifa + peajes + bcf + impuesto + congestión + aeropuerto con nulo = 0 + propina) y es {r['total_pagado']}."
    assert str(r["pickup_date"]) == "2024-01-10", "pickup_date debe ser la fecha de pickup_datetime."

    # 2. cada tipo de viaje imposible se descarta
    casos = [("distancia 0 mi", dict(distancia=0.0)),
             ("distancia de exactamente 100 mi (el límite es exclusivo)", dict(distancia=100.0)),
             ("tarifa base 0", dict(tarifa=0.0)),
             ("duración de 0 minutos", dict(segundos=0)),
             ("duración de 181 minutos", dict(segundos=181 * 60)),
             ("sin zona de recogida", dict(pu=None))]
    for descripcion, cambios in casos:
        n = f(spark.createDataFrame([_viaje(**cambios)], COLS_ENTRADA)).count()
        assert n == 0, f"No descartaste el viaje con {descripcion}."

    # 3. sobre la muestra real da lo mismo que silver.limpiar_viajes del pipeline
    mia, ref = f(muestra), silver.limpiar_viajes(muestra, MESES_LAB)
    assert mia.columns == ref.columns, f"Las columnas deben ser {ref.columns}, en ese orden; tienes {mia.columns}."
    assert mia.count() == ref.count(), f"Sobre la muestra conservaste {mia.count()} viajes y el pipeline conserva {ref.count()}."
    sobran = redondear_total(mia).exceptAll(redondear_total(ref)).count()
    faltan = redondear_total(ref).exceptAll(redondear_total(mia)).count()
    assert sobran == 0 and faltan == 0, "Las filas conservadas coinciden en número pero algún valor es distinto al del pipeline (revisa duración y total_pagado)."


verificar("E1 limpiar_tipos", limpiar_tipos, prueba_e1)

### Ejercicio 2 · Silver: quitar los duplicados usando la huella

Escribes tu código en la Celda 11, la Celda 12 lo verifica y la Celda 13 muestra su plan.

**Qué hacer.** Completa `quitar_duplicados(df)`: debe devolver `df` dejando **una sola fila por cada valor de `_hash_fila`**, con las mismas columnas que tenía. Hay dos maneras: `df.dropDuplicates(["_hash_fila"])`, o numerar las filas de cada huella con `row_number()` sobre una ventana `Window.partitionBy("_hash_fila").orderBy(...)` y quedarte con la número 1 (en ese caso recuerda quitar la columna auxiliar).

**Por qué importa, y por qué es un shuffle.** Las filas repetidas llegan por dos caminos: la fuente publica el mismo viaje dos veces, o una ingesta se repite y deja el mismo dato en dos lotes, como viste con el clima. La huella reduce la pregunta «¿estas dos filas son iguales?» a comparar un solo texto. Aun así, detectar un duplicado exige reunir las filas que tienen la misma huella, y esas filas pueden estar en cualquier partición: Spark tiene que repartir las filas por huella, es decir, hacer un shuffle. Después de la prueba, la Celda 13 imprime el plan para que veas el `Exchange`. Cuál de las dos filas repetidas se conserva no importa aquí, porque dos filas con la misma huella tienen idénticas sus columnas originales; solo difieren los metadatos de la ingesta. Una aclaración sobre el repositorio: el `silver.py` que está en `src/` todavía no quita duplicados. Esta función es justamente esa pieza, y entra al pipeline en la clase 6. Hoy no hay nada que quitar en el pipeline porque silver lee un solo lote; la huella importa cuando se juntan lotes o cuando la fuente repite filas.

<details>
<summary>Pista 1: una sola línea</summary>

`dropDuplicates` recibe una lista con las columnas que definen que dos filas son iguales. Aquí esa lista tiene una sola columna: la huella.

</details>

<details>
<summary>Pista 2: la versión con ventana</summary>

Si prefieres la ventana, numera las filas de cada huella con `F.row_number().over(Window.partitionBy("_hash_fila").orderBy("_hash_fila"))`, conserva solo la fila número 1 y quita la columna auxiliar con `drop`. La prueba exige que las columnas de salida sean las mismas que las de entrada.

</details>

In [ ]:
# Celda 11
def quitar_duplicados(df: DataFrame) -> DataFrame:
    """Una fila por cada valor de `_hash_fila`; conserva las columnas de `df`."""
    resultado = df.dropDuplicates(["_hash_fila"])
    # Alternativa con ventana (también es un shuffle):
    #   ventana = Window.partitionBy("_hash_fila").orderBy("_hash_fila")
    #   resultado = df.withColumn("_n", F.row_number().over(ventana)).where("_n = 1").drop("_n")
    return resultado

In [ ]:
# Celda 12
def prueba_e2(f):
    chico = spark.createDataFrame([("a", "h1", "x"), ("a", "h1", "y"), ("b", "h2", "z"), ("c", "h1", "w")],
                                  "k string, _hash_fila string, otro string")
    r = f(chico)
    assert r.count() == 2, f"De 4 filas con 2 huellas distintas deberían quedar 2 y quedaron {r.count()}."
    assert r.columns == chico.columns, f"Debes conservar las columnas {chico.columns} y devolviste {r.columns} (¿dejaste una columna auxiliar?)."

    real = f(muestra_dup_h)
    esperadas = muestra_dup_h.select("_hash_fila").distinct().count()
    assert real.count() == esperadas, f"La muestra tiene {esperadas} huellas distintas y devolviste {real.count()} filas."
    assert real.select("_hash_fila").distinct().count() == real.count(), "Todavía quedan huellas repetidas."
    assert real.columns == muestra_dup_h.columns, "Las columnas de la salida deben ser las mismas que las de la entrada."
    assert real.count() < muestra_dup_h.count(), "La muestra tiene filas repetidas a propósito y no quitaste ninguna."


verificar("E2 quitar_duplicados", quitar_duplicados, prueba_e2)

In [ ]:
# Celda 13
if RESULTADO_LAB.get("E2 quitar_duplicados") == "OK":
    print(f"Filas con repetidos: {fmt(muestra_dup_h.count())} -> sin repetidos: {fmt(quitar_duplicados(muestra_dup_h).count())}")
    print("\nPlan de tu función (busca el Exchange: es el shuffle que junta las filas con la misma huella):")
    mostrar_plan(quitar_duplicados(muestra_dup_h), ["Exchange", "HashAggregate", "Window", "Sort"])
else:
    print("Resuelve y verifica el ejercicio 2 para ver su plan.")

**Qué observar.** Hay un `Exchange hashpartitioning(_hash_fila...)`: es el shuffle que reparte las filas por huella para poder compararlas. Si usaste `dropDuplicates` verás también un `HashAggregate` parcial y uno final, igual que en el `GROUP BY` del segundo notebook; con la ventana verás `Sort` y `Window`. Pregúntate qué pasaría con un duplicado cuyas dos copias cayeran en particiones distintas si no hubiera shuffle: no se detectaría.

### Ejercicio 3 · Silver: unir las zonas con broadcast

Escribes tu código en la Celda 14 y la Celda 15 lo verifica.

**Qué hacer.** Completa `unir_zonas(trips, zonas)`. Debe unir cada viaje con su zona de recogida (`pu_location_id` contra `LocationID`) con un **`LEFT JOIN`** y pidiendo el **broadcast** de la tabla de zonas, y calcular la columna `zona`: el nombre de la zona, o `Zona <id>` cuando el identificador no está en la tabla. La función ya prepara `zonas_sel` con los nombres de columna que usa silver.

**Por qué importa.** Un `LEFT JOIN` no pierde viajes: los que no encuentran zona se quedan con un nombre genérico en vez de desaparecer, y eso se comprueba con el conteo. El broadcast evita mover los viajes por un shuffle, como viste en el segundo notebook. La prueba apaga el broadcast automático de Spark para comprobar que **tú** lo pediste explícitamente.

<details>
<summary>Pista 1: el join con broadcast</summary>

El `join` recibe, en este orden, el otro DataFrame, la columna por la que se unen y el tipo de unión. Aquí la columna se llama igual en los dos lados (`pu_location_id`), así que basta su nombre, y el tipo es `"left"` para no perder viajes. Para pedir el broadcast, envuelve `zonas_sel`, la tabla pequeña, con `F.broadcast(...)`.

</details>

<details>
<summary>Pista 2: el nombre de la zona</summary>

Para `zona` necesitas dos piezas. La primera es el texto genérico, `F.concat(F.lit("Zona "), <el identificador como texto>)`; el identificador es un entero, así que conviértelo con `cast("string")`. La segunda es `F.coalesce`, para usar el nombre de la zona (`zone_name`) y, solo si es nulo, ese texto genérico.

</details>

In [ ]:
# Celda 14
def unir_zonas(trips: DataFrame, zonas: DataFrame) -> DataFrame:
    """LEFT JOIN con las zonas (broadcast) y columna `zona` (nombre, o 'Zona <id>' si no existe)."""
    zonas_sel = zonas.select(
        F.col("LocationID").cast("int").alias("pu_location_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone_name"),
    )
    unido = trips.join(F.broadcast(zonas_sel), "pu_location_id", "left")
    zona = F.coalesce(F.col("zone_name"), F.concat(F.lit("Zona "), F.col("pu_location_id").cast("string")))
    return unido.withColumn("zona", zona).drop("zone_name")

In [ ]:
# Celda 15
def prueba_e3(f):
    chicos = spark.createDataFrame([(132, "a"), (9999, "b"), (1, "c")], "pu_location_id int, x string")
    umbral = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
    try:
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")     # sin broadcast automático
        res = f(chicos, zonas)
        plan = texto_plan(res)
        filas = {}
        for r in res.collect():
            filas[r["pu_location_id"]] = r
    finally:
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", umbral)
    assert len(filas) == 3, f"Un LEFT JOIN no debe perder viajes: entraron 3 y salieron {len(filas)}."
    assert filas[132]["zona"] == "JFK Airport" and filas[132]["borough"] == "Queens", "La zona 132 debe ser «JFK Airport» del borough «Queens»."
    assert filas[1]["zona"] == "Newark Airport", "La zona 1 debe ser «Newark Airport»."
    assert filas[9999]["zona"] == "Zona 9999", f"Una zona que no está en la tabla debe llamarse «Zona 9999» y se llama {filas[9999]['zona']!r}."
    assert filas[9999]["borough"] is None, "Para una zona inexistente, borough debe quedar nulo."
    assert "zone_name" not in res.columns, "La columna auxiliar zone_name no debe quedar en la salida."
    assert "BroadcastHashJoin" in plan, "El plan no usa BroadcastHashJoin: pide el broadcast de la tabla de zonas con F.broadcast(...)."
    # sobre la muestra: no se pierde ningún viaje
    limpio = silver.limpiar_viajes(muestra, MESES_LAB)
    assert f(limpio, zonas).count() == limpio.count(), "Sobre la muestra, la unión cambió el número de viajes."


verificar("E3 unir_zonas", unir_zonas, prueba_e3)

### Ejercicio 4 · Gold: agregar por día, zona y empresa, y por qué los promedios se ponderan

Escribes tu código en la Celda 16 (las dos funciones están en la misma celda) y la Celda 17 las verifica.

**Qué hacer, primera parte.** Completa `agregar_gold(silver)`: una fila por `fecha` (la `pickup_date`), `zona` y `empresa`, con `viajes` (cuántas filas), `ingreso_total` (suma de `total_pagado`, redondeada a 2 decimales), `propina_promedio` y `distancia_promedio` (promedios de `propina` y `distancia_mi`, redondeados a 2 decimales) y `lluvia_mm` (la precipitación del día, que es la misma para todos los viajes de ese día: `max`). Las columnas de salida son las de `COLUMNAS`, en ese orden. Es lo que hace `gold.metricas_diarias_zona`.

**Por qué importa.** Gold es lo que consume Snowflake: una tabla pequeña, con un grano claro (la clave `fecha, zona, empresa` no se repite) y sin que nadie vuelva a tocar los viajes individuales. Es una transformación ancha: un `groupBy`, con sus dos fases.

**Qué hacer, segunda parte.** Un día querrás una tabla más gruesa, por fecha y zona sin separar por empresa, y la tentación es promediar los promedios de gold. Eso es un error: si Uber hizo 8 viajes con propina media de 1 dólar y Lyft hizo 2 con propina media de 2, la propina media de la zona no es 1.5 sino `(8·1 + 2·2) / 10 = 1.2`. Un promedio de promedios trata igual a una empresa con 2 viajes que a otra con 8. La forma correcta es **ponderar por viajes**. Completa `resumen_por_zona(gold)`, que agrupa un gold por `fecha` y `zona` y calcula `propina_promedio` y `distancia_promedio` ponderando por `viajes`, y suma `viajes` e `ingreso_total`. `lluvia_mm` sigue siendo `max`. La prueba compara tu resultado con el promedio calculado directamente desde los viajes.

<details>
<summary>Pista 1: `agregar_gold`</summary>

Agrupa por tres columnas: la fecha (`pickup_date`, que en la salida se llama `fecha`; renómbrala con `alias` dentro del `groupBy`), `zona` y `empresa`. Dentro de `agg` van cinco expresiones, cada una con su `alias`: `F.count(F.lit(1))` cuenta las filas, `F.sum` suma `total_pagado`, `F.avg` promedia `propina` y `distancia_mi`, y `F.max` toma la lluvia. Los tres montos van con `F.round(..., 2)`.

</details>

<details>
<summary>Pista 2: `resumen_por_zona`</summary>

Un promedio ponderado por viajes es la suma de (promedio × viajes) dividida entre la suma de viajes. Escríbelo en dos partes: el numerador, `F.sum(F.col("propina_promedio") * F.col("viajes"))`, y el denominador, la variable `viajes` que ya está definida en la función. Redondea el cociente a 2 decimales con `F.round`.

</details>

In [ ]:
# Celda 16
def agregar_gold(silver: DataFrame) -> DataFrame:
    """Métricas diarias por zona de recogida y empresa (mismo grano y columnas que gold.metricas_diarias_zona)."""
    agregado = silver.groupBy(F.col("pickup_date").alias("fecha"), "zona", "empresa").agg(
        F.count(F.lit(1)).alias("viajes"),
        F.round(F.sum("total_pagado"), 2).alias("ingreso_total"),
        F.round(F.avg("propina"), 2).alias("propina_promedio"),
        F.round(F.avg("distancia_mi"), 2).alias("distancia_promedio"),
        F.max("lluvia_mm").alias("lluvia_mm"),
    )
    return agregado.select(*COLUMNAS)


def resumen_por_zona(gold: DataFrame) -> DataFrame:
    """Gold por fecha y zona (sin separar por empresa), con promedios ponderados por viajes."""
    viajes = F.sum("viajes")
    propina_ponderada = F.round(F.sum(F.col("propina_promedio") * F.col("viajes")) / viajes, 2)
    distancia_ponderada = F.round(F.sum(F.col("distancia_promedio") * F.col("viajes")) / viajes, 2)
    return gold.groupBy("fecha", "zona").agg(
        viajes.alias("viajes"),
        F.round(F.sum("ingreso_total"), 2).alias("ingreso_total"),
        propina_ponderada.alias("propina_promedio"),
        distancia_ponderada.alias("distancia_promedio"),
        F.max("lluvia_mm").alias("lluvia_mm"),
    )

In [ ]:
# Celda 17
ESQUEMA_SILVER = ("pickup_date date, zona string, empresa string, total_pagado double, propina double, "
                  "distancia_mi double, lluvia_mm double")
ESQUEMA_GOLD = ("fecha date, zona string, empresa string, viajes long, ingreso_total double, "
                "propina_promedio double, distancia_promedio double, lluvia_mm double")


def prueba_e4(f):
    chico = spark.createDataFrame([
        (date(2024, 1, 1), "A", "Uber", 10.0, 1.0, 2.0, 3.5),
        (date(2024, 1, 1), "A", "Uber", 20.0, 3.0, 4.0, 3.5),
        (date(2024, 1, 1), "A", "Lyft", 8.0, 0.0, 1.0, 3.5),
        (date(2024, 1, 1), "B", "Uber", 5.0, 0.0, 1.0, 3.5),
        (date(2024, 1, 2), "A", "Uber", 7.0, 0.5, 6.0, None),
    ], ESQUEMA_SILVER)
    g = f(chico)
    assert g.columns == COLUMNAS, f"Las columnas deben ser {COLUMNAS}; tienes {g.columns}."
    filas = {}
    for r in g.collect():
        filas[(r["fecha"], r["zona"], r["empresa"])] = r
    assert len(filas) == 4, f"Con estos datos deben salir 4 filas (una por fecha, zona y empresa) y salieron {len(filas)}."
    a1 = filas[(date(2024, 1, 1), "A", "Uber")]
    assert a1["viajes"] == 2, "Dos viajes de Uber en la zona A el día 1: viajes debe ser 2."
    assert a1["ingreso_total"] == 30.0, "ingreso_total es la suma de total_pagado (10 + 20)."
    assert a1["propina_promedio"] == 2.0 and a1["distancia_promedio"] == 3.0, "Los promedios de propina y distancia de Uber en A deben ser 2.0 y 3.0."
    assert a1["lluvia_mm"] == 3.5, "lluvia_mm es la lluvia del día."
    assert filas[(date(2024, 1, 2), "A", "Uber")]["lluvia_mm"] is None, "Si el día no tiene dato de lluvia, lluvia_mm debe quedar nulo."
    # sobre la referencia de silver: igual que gold.metricas_diarias_zona
    mia, ref = f(silver_ref), gold.metricas_diarias_zona(silver_ref)
    assert mia.exceptAll(ref).count() == 0 and ref.exceptAll(mia).count() == 0, "Sobre la referencia de silver, tu resultado difiere del de gold.metricas_diarias_zona."


def prueba_e4_ponderado(f):
    chico = spark.createDataFrame([
        (date(2024, 1, 1), "A", "Uber", 8, 40.0, 1.0, 3.0, 3.5),
        (date(2024, 1, 1), "A", "Lyft", 2, 30.0, 2.0, 8.0, 3.5),
    ], ESQUEMA_GOLD)
    r = f(chico).collect()
    assert len(r) == 1, "Las dos filas (Uber y Lyft) deben juntarse en una sola por fecha y zona."
    r = r[0]
    assert r["viajes"] == 10 and r["ingreso_total"] == 70.0, "viajes e ingreso_total se suman: 10 y 70.0."
    assert abs(r["propina_promedio"] - 1.2) < 1e-9, f"propina_promedio debería ser (8·1 + 2·2) / 10 = 1.2 y es {r['propina_promedio']}: pondera por viajes, no promedies los promedios."
    assert abs(r["distancia_promedio"] - 4.0) < 1e-9, f"distancia_promedio debería ser (8·3 + 2·8) / 10 = 4.0 y es {r['distancia_promedio']}."
    # contra el promedio calculado directamente desde los viajes
    mio = f(gold.metricas_diarias_zona(silver_ref))
    directo = silver_ref.groupBy(F.col("pickup_date").alias("fecha"), "zona").agg(
        F.count(F.lit(1)).alias("n"), F.avg("propina").alias("p"), F.avg("distancia_mi").alias("d"))
    cruce = mio.join(directo, ["fecha", "zona"])
    assert cruce.count() == directo.count() == mio.count(), "Deben salir exactamente las combinaciones fecha-zona de silver."
    malos = cruce.where((F.col("viajes") != F.col("n"))
                        | (F.abs(F.col("propina_promedio") - F.col("p")) > 0.011)
                        | (F.abs(F.col("distancia_promedio") - F.col("d")) > 0.011)).count()
    assert malos == 0, f"En {malos} combinaciones fecha-zona el promedio no coincide con el calculado directo desde los viajes (tolerancia de redondeo: 0.011)."


verificar("E4a agregar_gold", agregar_gold, prueba_e4)
verificar("E4b resumen_por_zona", resumen_por_zona, prueba_e4_ponderado)

### Tu pipeline completo contra el del repositorio

Cuando los ejercicios 1, 2, 3, 4a y 4b dicen `OK`, ya puedes encadenar tus funciones en el mismo orden que usa el pipeline: huella de los viajes, duplicados, limpieza de tipos, zonas, clima y agregación a gold. La Celda 19 las encadena sobre la muestra con repetidos y luego compara el resultado, fila por fila y por su llave, con el que produce el código del repositorio (`silver.limpiar_viajes`, `silver.enriquecer`, `gold.metricas_diarias_zona`) sobre los mismos datos y el mismo clima. La Celda 18 define la función que hace esa comparación.

El clima de la cadena sale del bronze que ingestaste en la clase 5. Si en tu carpeta hay lotes del clima de enero, se leen todos juntos, se quitan los duplicados por huella y queda una fila por día. Si no los hay, la cadena usa el clima del repositorio (el archivo `tests/data/open_meteo_sample.json`, con los 31 días de enero) y lo avisa al imprimir, para que llegues a la comparación de todos modos. En los dos casos el pipeline del repositorio recibe el mismo clima que el tuyo, así que la comparación es justa.

Los montos se comparan con una tolerancia de un centésimo: sumar decimales en distinto orden cambia la última cifra, y si un promedio cae justo en la mitad, el redondeo puede ir hacia un lado o hacia el otro. Si coinciden, tus funciones hacen lo mismo que las del pipeline y se pueden llevar a `src/`. La Celda 20 resume el resultado en una tabla de lluvia y viajes por día.

In [ ]:
# Celda 18
def comparar_gold(mio, referencia):
    """Compara dos gold por su llave (fecha, zona, empresa): viajes y lluvia iguales, montos iguales salvo el redondeo."""
    claves = ["fecha", "zona", "empresa"]
    montos = ["ingreso_total", "propina_promedio", "distancia_promedio"]
    mio_r = mio
    ref_r = referencia
    for nombre in ["viajes", "lluvia_mm"] + montos:
        mio_r = mio_r.withColumnRenamed(nombre, nombre + "_mio")
        ref_r = ref_r.withColumnRenamed(nombre, nombre + "_ref")
    cruce = mio_r.join(ref_r, claves, "full_outer").cache()

    sin_pareja = cruce.where(F.col("viajes_mio").isNull() | F.col("viajes_ref").isNull()).count()
    viajes_distintos = cruce.where(F.col("viajes_mio") != F.col("viajes_ref")).count()
    lluvia_distinta = cruce.where(~F.col("lluvia_mm_mio").eqNullSafe(F.col("lluvia_mm_ref"))).count()

    # Un monto puede diferir en un centésimo: sumar decimales en otro orden cambia la última cifra
    # y, si el valor cae justo en la mitad, el redondeo va hacia el otro lado.
    hay_diferencia = F.lit(False)
    for nombre in montos:
        diferencia = F.abs(F.col(nombre + "_mio") - F.col(nombre + "_ref"))
        hay_diferencia = hay_diferencia | (diferencia > 0.011)
    montos_distintos = cruce.where(hay_diferencia).count()
    cruce.unpersist()
    return {"sin_pareja": sin_pareja, "viajes_distintos": viajes_distintos,
            "montos_distintos": montos_distintos, "lluvia_distinta": lluvia_distinta}

In [ ]:
# Celda 19
ejercicios_de_la_cadena = [
    "E1 limpiar_tipos", "E2 quitar_duplicados",
    "E3 unir_zonas", "E4a agregar_gold", "E4b resumen_por_zona",
]
pendientes = []
for nombre in ejercicios_de_la_cadena:
    if RESULTADO_LAB.get(nombre) != "OK":
        pendientes.append(nombre)

if pendientes:
    print("Todavía no: resuelve y verifica los ejercicios", pendientes, "(deben decir OK; ejecuta también las celdas que falten).")
else:
    # ¿Hay un lote completo del clima de enero en el bronze de tu clase 5?
    try:
        pio.lote_mas_reciente(str(CARPETA_BRONZE), TABLA_CLIMA_CLASE5, "2024-01")
        hay_clima_de_la_clase_5 = True
    except pio.LoteNoEncontrado:
        hay_clima_de_la_clase_5 = False

    if hay_clima_de_la_clase_5:
        # Tu clima: se leen todos los lotes de enero juntos y se quitan los repetidos por huella.
        carpeta_mes = CARPETA_BRONZE / TABLA_CLIMA_CLASE5 / "mes=2024-01"
        bronze_clima = spark.read.parquet(str(carpeta_mes)).drop("lote")
        clima_lab = clima_diario(quitar_duplicados(bronze_clima))
        print("Filas del clima en bronze (todos los lotes):", bronze_clima.count(), "| sin repetidos:", clima_lab.count())

        # El pipeline del repositorio recibe el mismo clima, leído solo del lote más reciente, como lo hace silver.
        lote_reciente = pio.dir_lote_reciente(str(CARPETA_BRONZE), TABLA_CLIMA_CLASE5, "2024-01")
        clima_repo = clima_diario(spark.read.parquet(lote_reciente))
    else:
        # Sin la ingesta de la clase 5 no hay bronze de clima: los dos lados usan el clima de pruebas del repositorio.
        print("No hay clima en tu bronze de la clase 5: la cadena usa el clima del repositorio (tests/data/open_meteo_sample.json).")
        clima_lab = clima_pruebas
        clima_repo = clima_pruebas

    with cronometro("tu pipeline: huella -> duplicados -> tipos -> zonas -> clima -> gold"):
        viajes_con_huella = muestra_dup.withColumn("_hash_fila", calcular_hash_fila(muestra_dup.columns))
        silver_lab = unir_clima(unir_zonas(limpiar_tipos(quitar_duplicados(viajes_con_huella)), zonas), clima_lab)
        gold_lab = agregar_gold(silver_lab).cache()
        n_gold = gold_lab.count()

    # El del repositorio, sobre los mismos datos. El silver.py del repositorio no quita duplicados (esa pieza es tu
    # ejercicio 2), así que aquí se quitan a mano con dropDuplicates() para que los dos lados partan de lo mismo.
    silver_repo = silver.enriquecer(silver.limpiar_viajes(muestra.dropDuplicates(), MESES_LAB), zonas, clima_repo)
    referencia = gold.metricas_diarias_zona(silver_repo)

    print(f"Filas de tu gold: {fmt(n_gold)} | filas del gold del repo: {fmt(referencia.count())} | viajes: {fmt(gold_lab.agg(F.sum('viajes')).first()[0])}")
    comparacion = comparar_gold(gold_lab, referencia)
    print("Filas sin pareja (en uno y no en el otro):", comparacion["sin_pareja"])
    print("Filas con distinto número de viajes      :", comparacion["viajes_distintos"])
    print("Filas con montos que difieren más de 0.011:", comparacion["montos_distintos"])
    print("Filas con distinta lluvia                :", comparacion["lluvia_distinta"])
    exactas_distintas = gold_lab.exceptAll(referencia).count()
    print("Filas que difieren en el último decimal por el redondeo:", exactas_distintas)
    print("Tu gold es equivalente al del pipeline del repo:", sum(comparacion.values()) == 0)

In [ ]:
# Celda 20
if "gold_lab" in globals():
    diario = (gold_lab.groupBy("fecha").agg(F.sum("viajes").alias("viajes"), F.max("lluvia_mm").alias("lluvia_mm"))
              .orderBy("fecha").toPandas())
    print(diario.to_string(index=False))
    print("\nCorrelación día a día entre lluvia (mm) y viajes de la muestra:", round(float(diario["viajes"].corr(diario["lluvia_mm"])), 3))
else:
    print("Primero termina la celda anterior.")

**Qué observar.** Con los ejercicios de la cadena en `OK`, tu gold debe ser equivalente al del pipeline: ninguna fila sin pareja, los mismos viajes y la misma lluvia en cada fila, y montos que solo pueden diferir en el último decimal. Esta tabla es el primer vistazo a la pregunta del curso, con tus propias funciones. Si la cadena usó el clima del repositorio porque no tienes la ingesta de la clase 5, recuerda que ese archivo es de pruebas, con valores de ejemplo y no el clima real de enero: la correlación solo comprueba que la cadena funciona de punta a punta, y el clima real es el que trae tu ingesta. No es una conclusión: es **un mes**, **una muestra del 1 %** y una correlación entre dos series de 31 puntos, sin controlar nada más (el día de la semana pesa mucho en los viajes y se mezcla con la lluvia). Sirve para comprobar que la tubería de datos funciona de punta a punta; contestar la pregunta de verdad es lo que harás en el Taller 2, con el año completo y con cuidado.

### Cuándo cerrar Spark

La sección siguiente corre el pipeline desde la terminal, y los tests de `make test` también arrancan su propio Spark. En esta máquina solo cabe un proceso de Spark a la vez, así que **antes de pasar a la terminal debes cerrar el de este notebook** con la Celda 21. Después de ejecutarla, las celdas que usan `spark` dejan de funcionar hasta que reinicies el kernel y las vuelvas a correr desde la sección 0.

In [ ]:
# Celda 21
muestra.unpersist()
muestra_dup_h.unpersist()
silver_ref.unpersist()
spark.stop()
print("Spark cerrado. Ya puedes usar la terminal: make test, pipeline.run, etc.")

## 3. «A mano primero»: el mismo pipeline desde la terminal, y luego con `make`

Todo lo que hiciste en este notebook corre también como un programa, sin notebook. La regla de trabajo del curso es la misma cada vez que aparece una herramienta: primero el comando completo escrito a mano, con cada parte explicada; después el mismo comando con otro paso; y solo entonces el atajo.

### Paso 1. El comando completo

Abre una terminal en el Code Editor (menú *Terminal > New Terminal*) y ejecuta:

```bash
cd ~/bigdata && PYTHONPATH=src .venv/bin/python -m pipeline.run --step silver --env local --engine local --mes 2024-01
```

Parte por parte:

- `cd ~/bigdata` entra a la carpeta del repositorio. Todos los comandos de esta sección se ejecutan desde ahí; en otra carpeta no encontrarán ni `src` ni `.venv`.
- `PYTHONPATH=src` le dice a Python que busque paquetes en la carpeta `src`. El paquete `pipeline` vive ahí y no está instalado en el entorno, así que sin esto aparece `ModuleNotFoundError: No module named 'pipeline'`. (Es la misma ruta que agregaste en la Celda 2 con `sys.path.insert`.)
- `.venv/bin/python` es el Python del proyecto, el que tiene `pyspark` instalado. Escribir la ruta completa evita usar por error otro Python.
- `-m pipeline.run` ejecuta el módulo `src/pipeline/run.py` como programa. Es el punto de entrada de todos los pasos.
- `--step silver` elige el paso: `silver` limpia y enriquece (tus ejercicios 1, 2 y 3 hacen esto mismo; la ingesta de la clase 5 es el paso `bronze`, que en el pipeline corre en la nube). Los otros valores son `gold`, `all`, y `bronze` o `muestra`, que solo corren en la nube.
- `--env local` elige el archivo de configuración `src/pipeline/config/local.yaml`: lee la muestra de `./lake/muestra` y escribe en `./lake/silver`.
- `--engine local` elige el motor: Spark `local[*]` con 2 GB de driver, la misma sesión que arrancaste en la Celda 5.
- `--mes 2024-01` procesa solo ese mes.

**La primera vez** el programa copia solo la muestra a `./lake/muestra`: la busca en tu bucket y, si ahí no hay una con el formato nuevo, la baja del bucket público del curso. La línea `Muestra descargada` te dice de cuál la tomó. Las siguientes veces no hace falta. Para que busque primero en tu bucket, agrega al final del comando `--lake-bucket <tu bucket>`; la Celda 22 imprime el comando listo con tu bucket.

In [ ]:
# Celda 22
print(f"cd ~/bigdata && PYTHONPATH=src .venv/bin/python -m pipeline.run --step silver --env local --engine local --mes 2024-01 --lake-bucket {BUCKET}")

**Salida esperada.** Al ejecutarlo verás líneas de registro como estas, con tus números en lugar de los puntos suspensivos (lo verás al ejecutar):

```text
... INFO Paso=silver env=local engine=local meses=2024-01
... INFO Lectura viajes: ... filas (archivos ... MB, límite ninguno) desde .../lake/muestra, meses 2024-01
... INFO Limpieza: ... -> ... filas (... descartadas)
... INFO Calidad OK [silver]: ... filas
```

Léelas como una historia: cuántas filas entraron, cuántas sobrevivieron a la limpieza (la misma lógica que tu ejercicio 1) y que las reglas de calidad pasaron. Si una regla de calidad falla, el programa termina con un `QualityError` que dice la capa, la columna y cuántas filas están mal. Si Java no está disponible, el error aparece al arrancar Spark; si queda un Spark abierto en el notebook, verás que el programa se queda esperando o que falla por memoria: ciérralo con la celda de cierre de Spark.

**Ojo con el tamaño:** en el entorno `local`, el pipeline no lee el archivo de enero completo que usaste en el notebook, sino la **muestra** que se descargó sola la primera vez (de tu bucket o del bucket público del curso) en `~/bigdata/lake/muestra` (unos 8 mil viajes por mes). Por eso tarda segundos. La línea `Lectura viajes: ... desde .../lake/muestra` lo confirma. El mes completo se procesa en Glue, en la clase 7.


### Paso 2. Repetirlo con otro paso

Cambia solo el paso y vuelve a ejecutar. `gold` lee lo que escribió silver y escribe las métricas diarias por zona (tus ejercicios 4a y 4b hacen esto):

```bash
cd ~/bigdata && PYTHONPATH=src .venv/bin/python -m pipeline.run --step gold --env local --engine local --mes 2024-01
```

También puedes comprobar el código sin ejecutar el pipeline completo, con las pruebas y el verificador de estilo, siempre desde `~/bigdata`:

```bash
cd ~/bigdata && .venv/bin/python -m pytest tests/test_silver.py
cd ~/bigdata && .venv/bin/ruff check src
```

`pytest` ejecuta las pruebas de silver sobre datos de ejemplo (por eso también arranca Spark: no lo lances con el notebook abierto y encendido) y debe terminar con `passed`. `ruff check` revisa que el código siga las reglas de estilo y debe imprimir `All checks passed!`.

### Paso 3. Ahora, `make`

Abre el archivo `Makefile` de la raíz del repositorio. Cada *target* es un nombre para un comando que ya sabes escribir. Para comprobarlo, `make -n` muestra lo que se ejecutaría **sin ejecutarlo**:

```bash
cd ~/bigdata && make -n run-local STEP=silver MES=2024-01
```

Salida:

```text
STEP=silver MES=2024-01 LIMITE= bash scripts/run-local.sh
```

El target `run-local` solo pasa tus variables al script `scripts/run-local.sh`, y ese script arma los argumentos (`--step silver --env local --engine local --mes 2024-01`), calcula el bucket del lake si hace falta y ejecuta exactamente el `python -m pipeline.run` que escribiste a mano, con `PYTHONPATH` ya puesto. Ahora sí, el atajo:

```bash
cd ~/bigdata && make run-local STEP=silver MES=2024-01
```

Y lo mismo con las pruebas y el estilo: `make -n test` muestra `bash scripts/test.sh`, que ejecuta `ruff check`, `ruff format --check` y `python -m pytest`, es decir, los comandos del paso 2 y un poco más. Se ejecuta con `cd ~/bigdata && make test`. Recuerda: **cierra Spark en el notebook antes** (la Celda 21), porque `make test` y `make run-local` arrancan el suyo.

| Error común | Causa | Cómo evitarlo |
|---|---|---|
| `No rule to make target 'run-local'` | Ejecutaste `make` fuera de `~/bigdata` | Empieza siempre con `cd ~/bigdata &&` |
| `ModuleNotFoundError: No module named 'pipeline'` | Falta `PYTHONPATH=src` en el comando a mano | Escríbelo antes de `.venv/bin/python`, o usa `make` |
| `No hay muestra ... ni en tu bucket ni en el bucket público` | No se pudo leer ninguno de los dos orígenes (por ejemplo, sin internet) | Revisa tu conexión y vuelve a correr el comando |
| El programa falla al arrancar Spark o se queda esperando | El notebook sigue teniendo Spark abierto | Ejecuta la celda de cierre `spark.stop()` o reinicia el kernel |